In [ ]:
import pandas as pd
import numpy as np

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
import sqlite3

# Connect to database
conn = sqlite3.connect('../db.sqlite3')

# SQL query to join all 3 tables
query = """
SELECT 
    o.id as order_id,
    o.datetime as order_datetime,
    oi.id as order_item_id,
    oi.quantity,
    p.id as product_id,
    p.name as product_name,
    p.price as product_price
FROM restaurant_order o
JOIN restaurant_orderitem oi ON o.id = oi.order_id
JOIN restaurant_product p ON oi.product_id = p.id
"""

# Execute query and save to CSV
df_extracted = pd.read_sql_query(query, conn)
df_extracted.to_csv('../data.csv', index=False)

# Close connection
conn.close()

# Read the CSV file
df = pd.read_csv('../data.csv')

# Verify row count
print(f"Total rows: {len(df)}")
print(f"\nFirst few rows:")
df.head()

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
import matplotlib.pyplot as plt

# Group by product and sum quantities
product_quantities = df.groupby('product_name')['quantity'].sum().sort_values(ascending=False)

# Get top 10
top_10_quantity = product_quantities.head(10)

# Calculate percentages
percentages = (top_10_quantity / top_10_quantity.sum()) * 100

# Create pie chart
plt.figure(figsize=(10, 8))
plt.pie(percentages, labels=top_10_quantity.index, autopct='%1.1f%%', startangle=90)
plt.title('Top 10 Most Popular Products by Quantity')
plt.axis('equal')
plt.tight_layout()
plt.show()

print("Top 10 Products by Quantity:")
print(top_10_quantity)

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
# Calculate Item Price for each order item
df['item_price'] = df['product_price'] * df['quantity']

# Group by product and sum item prices
product_revenue = df.groupby('product_name')['item_price'].sum().sort_values(ascending=False)

# Get top 10
top_10_revenue = product_revenue.head(10)

# Calculate percentages
percentages_revenue = (top_10_revenue / top_10_revenue.sum()) * 100

# Create pie chart
plt.figure(figsize=(10, 8))
plt.pie(percentages_revenue, labels=top_10_revenue.index, autopct='%1.1f%%', startangle=90)
plt.title('Top 10 Products by Revenue')
plt.axis('equal')
plt.tight_layout()
plt.show()

print("Top 10 Products by Revenue:")
print(top_10_revenue)

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
# Convert order_datetime to datetime type
df['order_datetime'] = pd.to_datetime(df['order_datetime'])

# Extract hour from datetime
df['order_hour'] = df['order_datetime'].dt.hour

# Group by hour and sum item prices
hourly_revenue = df.groupby('order_hour')['item_price'].sum()

# Create bar chart
plt.figure(figsize=(12, 6))
plt.bar(hourly_revenue.index, hourly_revenue.values, color='steelblue', edgecolor='black')
plt.xlabel('Hour of Day')
plt.ylabel('Total Revenue')
plt.title('Restaurant Revenue by Hour of Day')
plt.xticks(range(0, 24))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("Revenue by Hour:")
print(hourly_revenue)

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
# Extract day of week from datetime (0=Monday, 6=Sunday)
df['order_day_of_week'] = df['order_datetime'].dt.dayofweek

# Group by day of week and sum item prices
daily_revenue = df.groupby('order_day_of_week')['item_price'].sum()

# Create labels for days
day_labels = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Create bar chart
plt.figure(figsize=(12, 6))
plt.bar(daily_revenue.index, daily_revenue.values, color='coral', edgecolor='black')
plt.xlabel('Day of Week')
plt.ylabel('Total Revenue')
plt.title('Restaurant Revenue by Day of Week')
plt.xticks(range(7), day_labels, rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("Revenue by Day of Week:")
for day_num, revenue in daily_revenue.items():
    print(f"{day_labels[day_num]}: {revenue:.2f}")